# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
%pip install -q --upgrade pypdf gradio openai python-dotenv

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [ ]:
import os

from pypdf import PdfReader

PDF_PATH = os.getenv("PDF_PATH", "attention_is_all_you_need.pdf")


def extract_text_from_pdf(pdf_path):
    """Lee un PDF y devuelve todo su texto como una sola cadena."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        # extract_text() puede devolver None en páginas sin texto
        text += (page.extract_text() or "") + "\n"
    return text


texto = extract_text_from_pdf(PDF_PATH)
print(texto[:500])

In [ ]:
from pathlib import Path
from pypdf import PdfReader
    
PDF_CANDIDATES = [
    Path("attention_is_all_you_need.pdf"),
    Path("1706.03762v7.pdf"),
]
PDF_PATH = next((path for path in PDF_CANDIDATES if path.exists()), PDF_CANDIDATES[0])


def extract_text_from_pdf(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages)


if PDF_PATH.exists():
    document_text = extract_text_from_pdf(PDF_PATH)
    print(f"PDF: {PDF_PATH}")
    print(f"Páginas: {len(PdfReader(str(PDF_PATH)).pages)}")
    print(f"Caracteres extraídos: {len(document_text):,}")
    print(document_text[:500])
else:
    document_text = ""
    print("No se encontró el PDF. Colócalo junto al notebook como attention_is_all_you_need.pdf.")

PDF: 1706.03762v7.pdf
Páginas: 15
Caracteres extraídos: 39,613
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


Resultados de la prueba: se extraen 39.600 caracteres. Los primeros 500 caracteres incluyen el aviso de permisos de Google, el título y los datos de los primeros autores. Las figuras (la arquitectura y las visualizaciones de atención de las últimas páginas) aportan fragmentos sueltos de texto, como etiquetas <EOS> y <pad> o palabras de las frases de ejemplo

## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.getenv("NVIDIA_API_KEY"):
    raise ValueError("Falta NVIDIA_API_KEY: créala en .env (no la imprimas).")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"

La verificación solo comprueba que la variable exista; nunca imprime la key.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("NVIDIA_API_KEY")
if api_key:
    print("NVIDIA_API_KEY cargada correctamente.")
else:
    print("NVIDIA_API_KEY no encontrada. La interfaz se podrá definir, pero no responderá hasta configurarla.")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key or "missing-api-key",
)
MODELO = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY no encontrada. La interfaz se podrá definir, pero no responderá hasta configurarla.


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [3]:
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper 'Attention Is All You Need'.

Responde únicamente usando la información del documento delimitado abajo. Si la respuesta
no aparece o no puede inferirse razonablemente del documento, responde exactamente:
'No encuentro esa información en el documento'. No inventes datos.

--- INICIO DEL DOCUMENTO ---
{document_text}
--- FIN DEL DOCUMENTO ---"""


def _text_from_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content)


def history_to_messages(history):
    messages = []
    for entry in history or []:
        if isinstance(entry, dict):
            role = entry.get("role")
            if role in {"user", "assistant"}:
                messages.append({
                    "role": role,
                    "content": _text_from_content(entry.get("content", "")),
                })
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_message, assistant_message = entry
            messages.append({"role": "user", "content": _text_from_content(user_message)})
            if assistant_message:
                messages.append({"role": "assistant", "content": _text_from_content(assistant_message)})
    return messages


def chat_con_documento(message, history, document_text):
    if not document_text:
        yield "No hay documento cargado. Verifica la ruta del PDF."
        return
    if not os.getenv("NVIDIA_API_KEY"):
        yield "Configura NVIDIA_API_KEY en el archivo .env antes de consultar el modelo."
        return

    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    response_text = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            response_text += content
            yield response_text

In [5]:
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" \
(Vaswani et al., 2017). Tu trabajo es responder preguntas sobre ese documento.

Reglas:
1. Responde ÚNICAMENTE con información que aparezca en el documento.
2. No uses conocimiento externo ni completes con suposiciones.
3. Si la respuesta no está en el documento, responde exactamente:
   "No encuentro esa información en el documento".
4. Responde en el mismo idioma de la pregunta y de forma clara y concisa.

<documento>
{document_text}
</documento>"""


def _content_to_text(content):
    """Gradio 6 puede entregar el contenido como str o como lista de bloques."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content)


def history_to_messages(history):
    """Convierte el historial de Gradio a mensajes con roles user/assistant.

    Soporta el formato "messages" (dicts con role/content) y el formato
    antiguo de tuplas [usuario, asistente].
    """
    messages = []
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            if role in ("user", "assistant"):
                messages.append(
                    {"role": role, "content": _content_to_text(item.get("content", ""))}
                )
        else:  # formato antiguo: (mensaje_usuario, respuesta_asistente)
            user_msg, bot_msg = item
            if user_msg:
                messages.append({"role": "user", "content": _content_to_text(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": _content_to_text(bot_msg)})
    return messages


def chat_con_documento(message, history, document_text):
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": _content_to_text(message)})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    respuesta = ""
    for chunk in stream:
        # Algunos chunks llegan sin choices o sin contenido: se ignoran
        if chunk.choices and chunk.choices[0].delta.content:
            respuesta += chunk.choices[0].delta.content
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [ ]:
import gradio as gr

document_text = extract_text_from_pdf(PDF_PATH)
n_paginas = len(PdfReader(PDF_PATH).pages)
print(f"PDF cargado: {n_paginas} páginas, {len(document_text):,} caracteres extraídos")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente de documentos: Attention Is All You Need",
    description=(
        "Haz preguntas sobre el paper del Transformer. "
        "Las respuestas se basan únicamente en el documento."
    ),
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
        ["¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"],
    ],
)

demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

Salida esperada al ejecutar: PDF cargado: 15 páginas, 39,600 caracteres extraídos.


In [4]:
import gradio as gr

if document_text:
    reader = PdfReader(str(PDF_PATH))
    print(f"Documento listo: {len(reader.pages)} páginas, {len(document_text):,} caracteres.")
else:
    print("Documento no disponible; revisa PDF_PATH antes de lanzar la interfaz.")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente sobre Attention Is All You Need",
    description="Responde únicamente con información extraída del paper.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
        ["¿Qué es GPT-4?"],
    ],
)

print("Ejecuta demo.launch(server_name='0.0.0.0', server_port=8080, show_error=True) para abrir Gradio.")

c:\Users\santi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documento listo: 15 páginas, 39,613 caracteres.
Ejecuta demo.launch(server_name='0.0.0.0', server_port=8080, show_error=True) para abrir Gradio.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

def estimar_contexto(texto):
    """Estimación aproximada: ~4 caracteres por token en inglés."""
    n_chars = len(texto)
    n_palabras = len(texto.split())
    return {
        "caracteres": n_chars,
        "palabras": n_palabras,
        "tokens_estimados": round(n_chars / 4),
    }

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

In [ ]:
def estimar_contexto(texto):
    """Estimación aproximada: ~4 caracteres por token en inglés."""
    n_chars = len(texto)
    n_palabras = len(texto.split())
    return {
        "caracteres": n_chars,
        "palabras": n_palabras,
        "tokens_estimados": round(n_chars / 4),
    }

A: 39.600 caracteres, 6.111 palabras, ~9.900 tokens estimados (estimación aproximada).
 
 ¿Por qué caracteres y tokens no son equivalentes? Un token es la unidad que el modelo procesa realmente, y se obtiene con un tokenizador que parte el texto en subpalabras. Una palabra común puede ser un solo token y una palabra rara, un número o una fórmula pueden dividirse en varios; el idioma también influye. Por eso el límite de contexto y el costo se miden en tokens, no en caracteres, y la relación de ~4 caracteres por token es solo una regla práctica

 B: 

In [ ]:
def texto_desde_subida(pdf_file, texto_por_defecto):
    """pdf_file llega como ruta (str) desde gr.File; None si no se subió nada."""
    if not pdf_file:
        return texto_por_defecto
    return "\n".join((p.extract_text() or "") for p in PdfReader(pdf_file).pages)


def construir_interfaz_con_subida(chat_fn_base, texto_por_defecto):
    def chat_con_subida(message, history, pdf_file):
        doc = texto_desde_subida(pdf_file, texto_por_defecto)
        yield from chat_fn_base(message, history, doc)

    return gr.ChatInterface(
        fn=chat_con_subida,
        title="Asistente con PDF dinámico",
        additional_inputs=[gr.File(label="Sube un PDF", file_types=[".pdf"])],
    )


demo = construir_interfaz_con_subida(chat_con_documento, document_text)

Si no se sube nada, usa el paper por defecto. Importante: un PDF escaneado no devuelve texto con pypdf; 

C:

In [ ]:
def extract_text_with_pages(pdf_path):
    reader = PdfReader(pdf_path)
    partes = []
    for i, page in enumerate(reader.pages, start=1):
        partes.append(f"[Página {i}]\n{page.extract_text() or ''}")
    return "\n\n".join(partes)


REGLA_CITAS = (
    "5. Cada respuesta debe terminar citando la(s) página(s) donde está la "
    "evidencia, con el formato (p. N). Las páginas están marcadas como [Página N]."
)


def build_system_prompt_con_citas(document_text):
    base = build_system_prompt(document_text)
    return base.replace("\n\n<documento>", "\n" + REGLA_CITAS + "\n\n<documento>", 1)

Se probó que el texto "N = 6 identical layers" queda dentro del marcador, que es donde está en el PDF. Para usarlo, pasa extract_text_with_pages(PDF_PATH) como documento y usa build_system_prompt_con_citas en lugar de build_system_prompt dentro de chat_con_documento.

D:

In [ ]:
import re

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "to", "is", "are", "and", "or", "what",
    "how", "does", "do", "for", "with", "by", "that", "this", "it", "as", "at",
}


def dividir_en_fragmentos(texto, tam=1500, solape=200):
    fragmentos, inicio = [], 0
    while inicio < len(texto):
        fragmentos.append(texto[inicio : inicio + tam])
        inicio += tam - solape
    return fragmentos


def tokenizar(texto):
    return [w for w in re.findall(r"\w+", texto.lower()) if w not in STOPWORDS]


def top_fragmentos(pregunta, fragmentos, k=3):
    palabras = set(tokenizar(pregunta))
    puntuados = []
    for i, frag in enumerate(fragmentos):
        conteo = tokenizar(frag)
        puntaje = sum(conteo.count(p) for p in palabras)
        puntuados.append((puntaje, i, frag))
    puntuados.sort(key=lambda x: (-x[0], x[1]))
    return [f for p, _, f in puntuados[:k] if p > 0]


def chat_con_rag_simple(message, history, document_text):
    fragmentos = dividir_en_fragmentos(document_text)
    relevantes = top_fragmentos(message, fragmentos, k=3)
    contexto = "\n---\n".join(relevantes)  # si no hay coincidencias queda vacío
    messages = [{"role": "system", "content": build_system_prompt(contexto)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})
    stream = client.chat.completions.create(
        model=MODELO, messages=messages, temperature=1.0, top_p=0.95, stream=True
    )
    respuesta = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            respuesta += chunk.choices[0].delta.content
            yield respuesta

Resultados de la prueba: el paper queda en 31 fragmentos. Para la pregunta el fragmento que contiene "N = 6" aparece entre los 3 más relevantes y el prompt pasa de 39.600 a unos 5.000 caracteres.


la búsqueda es por coincidencia exacta de palabras. El documento está en inglés, así que una pregunta en español solo encuentra fragmentos por palabras que se escriben igual en ambos idiomas.

## Reflexión final

**1. ¿Cuál es la limitación de enviar siempre el documento completo?**

Enviar siempre el documento completo aumenta el número de tokens, la latencia y el costo de cada consulta. Además, aunque la ventana de contexto sea grande, el modelo puede perder detalles relevantes entre mucho contenido y el sistema se vuelve menos escalable cuando se incorporan varios documentos.

RTA: Cada mensaje reenvía todo el paper más el historial, que crece en cada turno. Eso aumenta el costo y la latencia, y choca con el límite de la ventana de contexto: con un documento largo o varios documentos simplemente no cabe.

**2. ¿Por qué este ejercicio todavía no implementa RAG?**

Porque no existe una etapa de recuperación. El documento completo se inserta directamente en el prompt en cada consulta; en RAG primero se indexan los fragmentos y luego se recuperan solo los más relevantes para la pregunta antes de generar la respuesta.

RTA:Porque no hay recuperación, no se divide el documento en fragmentos, no se indexan ,y no se buscan los más relevantes para cada pregunta. RAG (Retrieval-Augmented Generation) envía al modelo solo los fragmentos pertinente, y por eso este prototipo solo sirve para documentos que caben en el contexto.

**3. ¿Cómo comprobarían que la respuesta proviene del documento y no del conocimiento previo del modelo?**

Usaría preguntas cuya respuesta esté explícita en el paper y preguntas trampa que no aparezcan en él. También exigiría citas de página o fragmentos de evidencia, compararía la respuesta con el texto original y verificaría que el sistema responda que no encuentra la información cuando el documento no la contiene.

RTA: Exigir citas con página y fragmento literal (mejora C) y verificar automáticamente que ese texto existe en el documento.
Preguntar cosas que no están en el documento (como GPT-4) y revisar que responda "No encuentro esa información".
Prueba contrafactual: modificar una copia del texto (por ejemplo cambiar "N = 6" por "N = 7") y comprobar que la respuesta cambia si siguiera diciendo 6, estaría usando su memoria.
Comparar con la respuesta del mismo modelo sin el documento en el prompt.
Bajar la temperature para que respete más la instrucción.

**4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el prompt?**

El documento puede contener instrucciones maliciosas (*prompt injection*) que intenten cambiar el comportamiento del asistente. También puede incluir información personal o confidencial, contenido falso, código peligroso o texto que exceda la ventana de contexto. Por eso conviene tratar el documento como datos no confiables, delimitarlo, validar archivos y controlar qué información puede devolver el sistema.

Prompt injection indirecta: el documento puede contener instrucciones que el modelo podría obedecer, porque no distingue bien entre datos e instrucciones.

Ruptura de delimitadores: un texto que incluya documento puede intentar escapar de la zona de datos.

Privacidad: documentos confidenciales o con datos personales se envían a un servicio externo (la API de NVIDIA).

Costo y disponibilidad: archivos enormes consumen tokens, dinero y pueden saturar el contexto.
Calidad: PDFs con tablas, fórmulas o escaneados se extraen mal y el modelo puede responder con datos incompletos o inventados.

Alucinaciones con apariencia de confianza: sin citas verificables, es difícil auditar si la respuesta es fiel.

**Resultado de la práctica:** el sistema extrae el paper con `pypdf`, lo incluye como contexto en un mensaje `system`, mantiene el historial de Gradio y genera respuestas en streaming. Esto demuestra document question answering, pero el siguiente paso para un sistema RAG sería fragmentar el documento, recuperar los fragmentos relevantes y citar sus fuentes.